# RLHF with PPO — Bangla নোটবুক

এটি `example.py`-এর একটি Bangla-অনুবাদিত, চালানোযোগ্য নোটবুক সংস্করণ।

RLHF with PPO

একটি সরলীকৃত PPO-স্টাইল policy-gradient training loop scratch থেকে
implement করে, যা সম্পূর্ণ RLHF পাইপলাইনের RL stage-র (README section 1)
বদলে একটি খেলনা "sequential token generation" কাজে প্রয়োগ করা হয়।
প্রক্রিয়াটি সম্পূর্ণ স্বচ্ছ রাখতে, "policy" T সংখ্যক sequence পজিশনের
প্রতিটিতে একটি ছোট vocabulary থেকে স্বাধীনভাবে একটি token বেছে নেয়
(একটি বাস্তব autoregressive language model-এর একটি ব্যাপক সরলীকরণ --
পূর্বে উৎপন্ন token-এর উপর কোনো conditioning নেই -- কিন্তু এটি এই
লেসনের বিষয় RL যন্ত্রপাতিটি হুবহু সংরক্ষণ করে: per-token reward, একটি
frozen reference policy-র বিরুদ্ধে per-token KL penalty, advantage
estimation, এবং PPO-র clipped surrogate objective)।

খেলনা reward function উৎপন্ন sequence-এ একটি নির্দিষ্ট "hack token"-
এর প্রতিটি উপস্থিতির জন্য +1 স্কোর দেয় -- একটি blind spot-যুক্ত reward
model-এর পরিবর্ত হিসেবে (README section 3): কোনো সীমাবদ্ধতা ছাড়া এই
reward সর্বাধিক করার অর্থ হলো সেই একটিমাত্র token spam করা, যেটি অবিকল
সে ধরনের অবক্ষয়িত, ভাষাসদৃশ নয় এমন আচরণ যা একটি বাস্তব reward model-এর
quirks-কে শোষণ করে বের করা যায়।

আমরা একই training loop দুবার চালাই: একবার frozen reference policy-র
বিরুদ্ধে KL penalty সহ (beta > 0), আর একবার এটি বন্ধ করে (beta = 0)
-- যাতে reward hacking-কে সততার সাথে পাশাপাশি পুনরুত্পাদন করা যায় সেই
প্রক্রিয়ার সাথে যা এটিকে প্রতিরোধ করে।

Runtime: CPU-তে কয়েক সেকেন্ড (সাধারণ অর্থে কোনো neural network training
নেই -- এটি একটি ছোট logits matrix সরাসরি optimize করে)।

চালান:
    নোটবুকের সব code cell উপরে থেকে নিচে চালান (Cell -> Run All)।

## কীভাবে চালাবেন

উপর থেকে নিচে (Cell → Run All) সব cell চালান। Runtime: CPU-তে কয়েক সেকেন্ড।

In [ ]:
"""RLHF with PPO

একটি সরলীকৃত PPO-স্টাইল policy-gradient training loop scratch থেকে
implement করে, যা সম্পূর্ণ RLHF পাইপলাইনের RL stage-র (README section 1)
বদলে একটি খেলনা "sequential token generation" কাজে প্রয়োগ করা হয়।
প্রক্রিয়াটি সম্পূর্ণ স্বচ্ছ রাখতে, "policy" T সংখ্যক sequence পজিশনের
প্রতিটিতে একটি ছোট vocabulary থেকে স্বাধীনভাবে একটি token বেছে নেয়
(একটি বাস্তব autoregressive language model-এর একটি ব্যাপক সরলীকরণ --
পূর্বে উৎপন্ন token-এর উপর কোনো conditioning নেই -- কিন্তু এটি এই
লেসনের বিষয় RL যন্ত্রপাতিটি হুবহু সংরক্ষণ করে: per-token reward, একটি
frozen reference policy-র বিরুদ্ধে per-token KL penalty, advantage
estimation, এবং PPO-র clipped surrogate objective)।

খেলনা reward function উৎপন্ন sequence-এ একটি নির্দিষ্ট "hack token"-
এর প্রতিটি উপস্থিতির জন্য +1 স্কোর দেয় -- একটি blind spot-যুক্ত reward
model-এর পরিবর্ত হিসেবে (README section 3): কোনো সীমাবদ্ধতা ছাড়া এই
reward সর্বাধিক করার অর্থ হলো সেই একটিমাত্র token spam করা, যেটি অবিকল
সে ধরনের অবক্ষয়িত, ভাষাসদৃশ নয় এমন আচরণ যা একটি বাস্তব reward model-এর
quirks-কে শোষণ করে বের করা যায়।

আমরা একই training loop দুবার চালাই: একবার frozen reference policy-র
বিরুদ্ধে KL penalty সহ (beta > 0), আর একবার এটি বন্ধ করে (beta = 0)
-- যাতে reward hacking-কে সততার সাথে পাশাপাশি পুনরুত্পাদন করা যায় সেই
প্রক্রিয়ার সাথে যা এটিকে প্রতিরোধ করে।

Runtime: CPU-তে কয়েক সেকেন্ড (সাধারণ অর্থে কোনো neural network training
নেই -- এটি একটি ছোট logits matrix সরাসরি optimize করে)।

চালান:
    নোটবুকের সব code cell উপরে থেকে নিচে চালান (Cell -> Run All)।
"""

import torch
import torch.nn.functional as F

torch.manual_seed(0)

## 1. Constant ও frozen reference policy

খেলনা "ভোকাবুলারি", sequence দৈর্ঘ্য এবং hack token; তারপর `make_reference_logits()` দিয়ে SFT-সদৃশ প্রারম্ভিক `pi_ref` তৈরি।

In [ ]:
VOCAB_SIZE = 6
SEQ_LEN = 8
HACK_TOKEN = VOCAB_SIZE - 1     # (খেলনা, hackable) reward function যে token-টিকে পুরস্কৃত করে
BATCH_SIZE = 256                # প্রতি rollout-এ episode ("উৎপন্ন sequence")
NUM_ITERS = 60                  # বাইরের rollout -> update cycle
PPO_EPOCHS = 4                  # প্রতি rollout batch-এ PPO update epoch
CLIP_EPS = 0.2                  # PPO-র trust-region clip পরিসর


def make_reference_logits():
    """Frozen reference policy pi_ref -- SFT মডেলের পরিবর্ত (README section 4)।
    প্রতিটি পজিশনে vocabulary-র উপর একটি হালকা non-uniform distribution
    দিয়ে initialize করা, যা "স্বাভাবিক, ইতিমধ্যে যুক্তিসঙ্গত" আচরণকে
    প্রতিনিধিত্ব করে যেকোনো RL fine-tuning-এর আগে।"""
    torch.manual_seed(42)
    return torch.randn(SEQ_LEN, VOCAB_SIZE) * 0.5


REFERENCE_LOGITS = make_reference_logits()

## 2. KL divergence, rollout এবং PPO update

`rollout()` বর্তমান policy থেকে খেলনা sequence-এর একটি batch নমুনা করে per-token reward − KL penalty ও একটি সরল baseline advantage গণনা করে; `ppo_update()` একই batch-এ PPO-র clipped surrogate objective-এর কয়েক epoch চালায়।

In [ ]:
def analytic_kl(policy_logits, reference_logits):
    """সঠিক KL( pi_theta(.|t) || pi_ref(.|t) ), সব T পজিশন জুড়ে যোগ করা।
    এখানে শুধুমাত্র MONITORING-এর জন্য ব্যবহৃত হয় যে policy কত দূরে
    সরে গেছে -- প্রশিক্ষণের সময় প্রয়োগ করা প্রকৃত per-token penalty
    rollout()-এ গণনা করা মানক sampled log-ratio estimator ব্যবহার করে,
    যেভাবে প্রকৃত RLHF implementation-গুলো করে (README section 4)।"""
    log_p = F.log_softmax(policy_logits, dim=-1)
    log_ref = F.log_softmax(reference_logits, dim=-1)
    p = log_p.exp()
    kl_per_position = (p * (log_p - log_ref)).sum(dim=-1)
    return kl_per_position.sum().item()


def rollout(policy_logits, beta):
    """বর্তমান policy থেকে খেলনা 'sequence'-এর একটি batch নমুনা করুন (এটি
    পরবর্তী update epoch-গুলির জন্য PPO-র 'পুরাতন' policy হয়ে যায়), এবং
    per-token reward-minus-KL-penalty এবং একটি সরল batch-mean baseline
    advantage গণনা করুন।

    ফেরত দেয়: sampled actions, old log-probs (detached), advantages (detached)।
    """
    with torch.no_grad():
        probs = F.softmax(policy_logits, dim=-1)                     # (T, V)
        dist = torch.distributions.Categorical(probs=probs.unsqueeze(0).expand(BATCH_SIZE, -1, -1))
        actions = dist.sample()                                       # (batch, T)

        old_log_probs = dist.log_prob(actions)                        # (batch, T)

        ref_probs = F.softmax(REFERENCE_LOGITS, dim=-1)
        ref_dist = torch.distributions.Categorical(probs=ref_probs.unsqueeze(0).expand(BATCH_SIZE, -1, -1))
        ref_log_probs = ref_dist.log_prob(actions)                    # (batch, T)

        raw_reward = (actions == HACK_TOKEN).float()                  # (batch, T) প্রতি hack-token হিটে 1
        kl_estimate = old_log_probs - ref_log_probs                   # per-token sampled KL estimate
        total_reward = raw_reward - beta * kl_estimate                # README section 4-র সূত্র, per token

        baseline = total_reward.mean(dim=0, keepdim=True)             # সরল variance-reduction baseline
        advantages = total_reward - baseline

    return actions, old_log_probs, advantages, raw_reward.sum(dim=1).mean().item()


def ppo_update(policy_logits, actions, old_log_probs, advantages, optimizer):
    """একই rollout batch-এ PPO clipped surrogate objective-এর (README
    section 5) কয়েক epoch -- অবিকল PPO-র মূল দক্ষতার কৌশল: একটি
    (একটি প্রকৃত LLM-এ, ব্যয়বহুল) rollout থেকে কয়েকটি gradient update
    বের করে নেওয়া।"""
    for _ in range(PPO_EPOCHS):
        probs = F.softmax(policy_logits, dim=-1)
        dist = torch.distributions.Categorical(probs=probs.unsqueeze(0).expand(BATCH_SIZE, -1, -1))
        new_log_probs = dist.log_prob(actions)

        ratio = torch.exp(new_log_probs - old_log_probs)
        surrogate1 = ratio * advantages
        surrogate2 = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * advantages
        loss = -torch.min(surrogate1, surrogate2).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

## 3. Training loop এবং চূড়ান্ত policy বর্ণনা

`train(beta, label)` পুরো rollout→update চক্র চালায় এবং history রাখে; `describe_final_policy()` শেখা policy-টি hacker token-এ কতটা probability mass বসিয়েছে তা দেখায়।

In [ ]:
def train(beta, label):
    policy_logits = REFERENCE_LOGITS.clone().detach().requires_grad_(True)
    optimizer = torch.optim.Adam([policy_logits], lr=0.1)

    history = []
    for it in range(1, NUM_ITERS + 1):
        actions, old_log_probs, advantages, avg_raw_reward = rollout(policy_logits.detach(), beta)
        ppo_update(policy_logits, actions, old_log_probs, advantages, optimizer)

        kl = analytic_kl(policy_logits.detach(), REFERENCE_LOGITS)
        history.append((it, avg_raw_reward, kl))

        if it % 10 == 0 or it == 1:
            print(f"  [{label}] iter {it:3d}   avg reward/episode = {avg_raw_reward:.3f} "
                  f"(max possible = {SEQ_LEN})   KL(policy || reference) = {kl:.3f}")

    return policy_logits.detach(), history


def describe_final_policy(policy_logits, label):
    probs = F.softmax(policy_logits, dim=-1)
    hack_prob = probs[:, HACK_TOKEN].mean().item()
    print(f"\n[{label}] average probability mass the policy places on the hack token "
          f"across all {SEQ_LEN} positions: {hack_prob:.3f}")
    print(f"[{label}] full probability distribution at position 0: "
          f"{[round(p, 3) for p in probs[0].tolist()]}")

## সম্পূর্ণ ডেমো চালানো

`main()` দুইটি রান তুলনা করে: KL penalty সহ (`beta=0.5`) সঠিক রেসিপি, এবং KL ছাড়া (`beta=0.0`) — ইচ্ছাকৃতভাবে reward hacking পুনরুত্পাদন করা হয়।

In [ ]:
def main():
    print("=" * 70)
    print("SETUP")
    print("=" * 70)
    print(f"Vocabulary size: {VOCAB_SIZE}, sequence length: {SEQ_LEN}, "
          f"'hack token' index: {HACK_TOKEN}")
    print("Toy reward = +1 for every occurrence of the hack token in the sequence --")
    print("a stand-in for a reward model with an exploitable blind spot (README section 3).")
    ref_probs_pos0 = F.softmax(REFERENCE_LOGITS[0], dim=-1)
    print(f"Reference (frozen SFT-like) policy's distribution at position 0: "
          f"{[round(p, 3) for p in ref_probs_pos0.tolist()]}")

    print("\n" + "=" * 70)
    print("1. TRAINING WITH THE KL PENALTY (beta = 0.5) -- the correct recipe")
    print("=" * 70)
    policy_with_kl, history_with_kl = train(beta=0.5, label="beta=0.5")
    describe_final_policy(policy_with_kl, "beta=0.5")

    print("\n" + "=" * 70)
    print("2. TRAINING WITH NO KL PENALTY (beta = 0.0) -- reward hacking, on purpose")
    print("=" * 70)
    policy_no_kl, history_no_kl = train(beta=0.0, label="beta=0.0")
    describe_final_policy(policy_no_kl, "beta=0.0")

    print("\n" + "=" * 70)
    print("3. COMPARISON")
    print("=" * 70)
    final_reward_kl = history_with_kl[-1][1]
    final_kl_kl = history_with_kl[-1][2]
    final_reward_nokl = history_no_kl[-1][1]
    final_kl_nokl = history_no_kl[-1][2]

    print(f"{'setting':>14}{'final avg reward':>20}{'final KL':>14}")
    print(f"{'beta=0.5':>14}{final_reward_kl:>20.3f}{final_kl_kl:>14.3f}")
    print(f"{'beta=0.0':>14}{final_reward_nokl:>20.3f}{final_kl_nokl:>14.3f}")

    print(f"\n-> With the KL penalty active, average reward rose from a near-random")
    print(f"   starting point to {final_reward_kl:.2f} out of a max of {SEQ_LEN}, while KL")
    print(f"   divergence from the reference policy stayed bounded at {final_kl_kl:.3f} --")
    print(f"   reward improved WITHOUT the policy drifting arbitrarily far from")
    print(f"   sensible (reference-like) behavior.")
    print(f"\n-> With NO KL penalty, reward reached {final_reward_nokl:.2f} -- HIGHER than the")
    print(f"   KL-constrained run -- but KL divergence exploded to {final_kl_nokl:.3f}, "
          f"{final_kl_nokl / max(final_kl_kl, 1e-6):.0f}x")
    print(f"   larger. The policy achieves this extra reward by collapsing almost all")
    print(f"   probability mass onto the single hack token at every position (see the")
    print(f"   'average probability mass on the hack token' lines above) -- exactly")
    print(f"   the degenerate, non-language-like reward hacking behavior the KL")
    print(f"   penalty exists to prevent. Higher reward here is NOT a better policy;")
    print(f"   it is Goodhart's law made concrete: a proxy (the reward model / toy")
    print(f"   reward function) stops tracking what it was meant to measure once")
    print(f"   optimization pressure against it is unconstrained.")

main()